# Music Annotation Project: Supervised Pilot Notebook 09

**Runtime:** CPU when confidence maps already exist. Use Notebook 06 on a GPU first for new pages.

This pilot produces risk-aware correction proposals and opens a supervised Gradio reviewer. It does not auto-accept, retrain, recalibrate or alter the frozen V7.7F threshold.

## Structural implementation contract

The functions in the canonical block are copied from the V7.8 rebuilt builder. Their source hash is recorded in every pilot manifest. The V7.8.2 equivalence audit defines the required regression stages: removal mask, Telea, geometry_t80, routing maps, connected components and predictors.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn joblib gradio pillow


In [ ]:
import cv2,json,hashlib,base64,io,platform,sys
import numpy as np
import pandas as pd
import joblib,gradio as gr
from pathlib import Path
from PIL import Image
from datetime import datetime,timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
PILOT_ROOT=ROOT/'production_pilot'
INCOMING=PILOT_ROOT/'incoming';DRAFT_MASKS=ROOT/'human_in_the_middle'/'draft_masks';CONFIDENCE_MAPS=ROOT/'human_in_the_middle'/'confidence_maps';DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit';REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit';REVIEWED_MASKS=PILOT_ROOT/'reviewed_masks'
OUTPUT_ROOT=PILOT_ROOT/'runs'
POLICY_RUN_OVERRIDE=None

MASK_DILATION=2;CROP_MARGIN=48;MIN_COMPONENT_AREA=20;GEOMETRY_THRESHOLD=80;BAND_WIDTH=4;CONTEXT_RING_RADIUS=18
LEGACY_ALIASES={'component_pixels_x':'component_pixels','component_pixels_y':'component_pixels'}
PUBLIC_LINK=True
for p in [INCOMING,DRAFT_AUDIT_DIR,REVIEW_AUDIT_DIR,REVIEWED_MASKS,OUTPUT_ROOT]:p.mkdir(parents=True,exist_ok=True)
RUN=OUTPUT_ROOT/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_pilot09');PROPOSALS=RUN/'proposals';FINAL=RUN/'final';MASK_OUT=RUN/'masks';ROUTING=RUN/'routing';AUDIT=RUN/'audit';TABLES=RUN/'tables'
for p in [RUN,PROPOSALS,FINAL,MASK_OUT,ROUTING,AUDIT,TABLES]:p.mkdir(parents=True,exist_ok=False)
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def immutable(original,candidate,mask):
 out=original.copy();out[mask>0]=candidate[mask>0];assert np.array_equal(out[mask==0],original[mask==0]);return out
def latest_complete(root,required):
 hits=[]
 if root.exists():
  for p in root.rglob('*'):
   if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None


## Canonical V7.8 structural functions: copied verbatim


In [ ]:
def binary(mask):return np.where(mask>0,255,0).astype(np.uint8)
def prepare_mask(seed,dilation=MASK_DILATION):
 mask=binary(seed)
 if dilation>0:mask=cv2.dilate(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
 return mask

def padded_components(mask):
 n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);h,w=mask.shape;boxes=[]
 for i in range(1,n):
  x,y,bw,bh,area=stats[i]
  if area>=8:boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
 changed=True
 while changed:
  changed=False;merged=[]
  while boxes:
   ax0,ay0,ax1,ay1=boxes.pop();rest=[]
   for bx0,by0,bx1,by1 in boxes:
    if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
    else:rest.append((bx0,by0,bx1,by1))
   boxes=rest;merged.append((ax0,ay0,ax1,ay1))
  boxes=merged
 return boxes

def telea_reconstruct(observed,mask):
 result=observed.copy()
 for x0,y0,x1,y1 in padded_components(mask):
  im=observed[y0:y1,x0:x1];mk=mask[y0:y1,x0:x1];cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA);local=result[y0:y1,x0:x1];local[mk>0]=cand[mk>0]
 return immutable(observed,result,mask)

def background_surface(observed,mask,sigma=45.0):
 visible=(mask==0)&(observed>=200);values=observed.astype(np.float32)*visible;weights=visible.astype(np.float32)
 num=cv2.GaussianBlur(values,(0,0),sigma);den=cv2.GaussianBlur(weights,(0,0),sigma);surface=num/np.maximum(den,1e-4);fallback=cv2.GaussianBlur(observed,(0,0),sigma)
 return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)
def outside_dark(observed,mask):return (observed<190)&(mask==0)
def support_maps(observed,mask):
 dark=outside_dark(observed,mask);h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0;v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
 n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);c=np.zeros_like(dark)
 for i in range(1,n):
  x,y,w,hg,a=stats[i]
  if 3<=a<=650 and w<=40 and hg<=40:c[lab==i]=True
 k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8)
 return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(c.astype(np.uint8),k)>0

def geometry_t80(observed,mask,telea):
 paper=background_surface(observed,mask);normalised=cv2.divide(observed,np.maximum(paper,1),scale=255);horizontal,vertical,compact=support_maps(observed,mask);m=mask>0
 preserve=m&(horizontal|vertical|compact)&(observed<200)
 remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(normalised>=GEOMETRY_THRESHOLD)
 ambiguous=m&~preserve&~remove
 result=observed.copy();result[remove]=paper[remove];result[ambiguous]=telea[ambiguous];result[preserve]=observed[preserve]
 return immutable(observed,result,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact,'paper':paper,'normalised':normalised}

def surrounding_features(observed,component,routes):
 ys,xs=np.where(component);x0,x1=xs.min(),xs.max()+1;y0,y1=ys.min(),ys.max()+1;area=int(component.sum());w=x1-x0;h=y1-y0
 kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*CONTEXT_RING_RADIUS+1,2*CONTEXT_RING_RADIUS+1));ring=(cv2.dilate(component.astype(np.uint8),kernel)>0)&~component
 vals=observed[ring];edges=cv2.Canny(observed,70,170)>0;depth=cv2.distanceTransform(component.astype(np.uint8),cv2.DIST_L2,5)
 def f(k):return float(np.sum(routes[k]&component)/area)
 return {'w':w,'h':h,'area':area,'aspect_ratio':max(w,h)/max(min(w,h),1),'fill_ratio':area/max(w*h,1),'touches_page_border':bool(x0==0 or y0==0 or x1==observed.shape[1] or y1==observed.shape[0]),'context_pixels':int(ring.sum()),'context_edge_density':float(edges[ring].mean()) if ring.any() else 0.0,'context_dark_density':float((vals<190).mean()) if vals.size else 0.0,'context_mean_intensity':float(vals.mean()) if vals.size else 255.0,'context_intensity_std':float(vals.std()) if vals.size else 0.0,'maximum_mask_depth':float(depth[component].max()),'horizontal_fraction':f('horizontal'),'vertical_fraction':f('vertical'),'compact_fraction':f('compact'),'preserve_fraction':f('preserve'),'remove_fraction':f('remove'),'ambiguous_fraction':f('ambiguous'),'reconstruction_dependency_fraction':f('ambiguous'),'component_pixels':area}


In [ ]:
CANONICAL_STRUCTURAL_SOURCE='17_V7_8_Rebuilt_Independent_Data_Builder_Cached_Registration.txt'
CANONICAL_STRUCTURAL_BLOCK_SHA256='cce24d65e8866b086956855b55e9a5104383c8b936629c5035354968c929b04c'
REUSE_REGISTRY={
 'binary':'V7.8 exact V7.1 dilation-2 block','prepare_mask':'V7.8 exact V7.1 dilation-2 block','padded_components':'V7.8 exact V7.1 dilation-2/Telea block','telea_reconstruct':'V7.8 exact V7.1 dilation-2/Telea block','background_surface':'V7.8 frozen V7.5 geometry_t80 block','outside_dark':'V7.8 frozen V7.5 geometry_t80 block','support_maps':'V7.8 frozen V7.5 geometry_t80 block','geometry_t80':'V7.8 frozen V7.5 geometry_t80 block','surrounding_features':'V7.8 frozen V7.5 surrounding evidence block','immutable':'V7.8 shared immutability helper'}
print('Canonical structural block SHA-256:',CANONICAL_STRUCTURAL_BLOCK_SHA256)
print(json.dumps(REUSE_REGISTRY,indent=2))


## Load and verify the frozen V7.7F policy bundle


In [ ]:
required=['artefacts/policy.json','artefacts/predictor_schema.json','artefacts/structural_model.joblib','artefacts/isotonic_calibrator.joblib','artefacts/replay_test.json']
POLICY_RUN=Path(POLICY_RUN_OVERRIDE) if POLICY_RUN_OVERRIDE else latest_complete(ROOT/'structural_policy_freeze_v7_7',required)
if POLICY_RUN is None:raise RuntimeError('Set POLICY_RUN_OVERRIDE to a complete V7.7F freeze run.')
P=POLICY_RUN/'artefacts';policy=json.loads((P/'policy.json').read_text());schema=json.loads((P/'predictor_schema.json').read_text());replay=json.loads((P/'replay_test.json').read_text())
checks={'replay':replay.get('passed') is True and replay.get('policy_id')==policy.get('policy_id'),'model_hash':policy['model_sha256']==sha256_file(P/'structural_model.joblib'),'calibrator_hash':policy['calibrator_sha256']==sha256_file(P/'isotonic_calibrator.joblib'),'schema_hash':policy['schema_sha256']==sha256_file(P/'predictor_schema.json')}
if not all(checks.values()):raise RuntimeError(f'Frozen policy integrity failure: {checks}')
model=joblib.load(P/'structural_model.joblib');calibrator=joblib.load(P/'isotonic_calibrator.joblib');predictors=schema['ordered_predictors'];threshold=float(policy['threshold'])
print('Policy:',policy['policy_id'],'threshold:',threshold,'checks:',checks)


## Mandatory V7.8.2-style development regression gate

This gate runs against the complete corrected V7.5 development reference used by the V7.8.2 audit. It does **not** use V7.8 independent pages and does not copy anything into the pilot input directory.

The reviewer cannot be created unless all of the following pass across every available development page:

- cached mask, Telea and `geometry_t80` arrays are present and shape-compatible;
- cached Telea and rebuilt `geometry_t80` are byte-for-byte immutable outside the mask;
- rebuilt `geometry_t80` exactly equals the V7.5 cached reference;
- all cached preserve/remove/ambiguous/horizontal/vertical/compact maps exist and match exactly;
- each retained connected component resolves to exactly one V7.5 evidence row;
- every shared predictor matches using the V7.8.2 tolerances;
- each retained component resolves to exactly one V7.7 development row.

The gate writes its full tables and verdict under the Pilot 09 run directory. Any failure raises `RuntimeError` before `/production_pilot/incoming` is enumerated and before Gradio is launched.


In [ ]:
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
V77_ROOT=ROOT/'structural_risk_calibration_v7_7'
V75_RUN_OVERRIDE=None
V77_DEVELOPMENT_RUN_OVERRIDE=None
REGRESSION_PAGE_ALLOWLIST=None  # Leave None for the mandatory complete development roster.
REGRESSION_MAX_PAGES=None       # Leave None for the mandatory complete development roster.
REGRESSION_ATOL=1e-9
REGRESSION_RTOL=1e-7
REGRESSION_DIR=RUN/'development_regression_gate'
REGRESSION_TABLES=REGRESSION_DIR/'tables'
REGRESSION_DIFFS=REGRESSION_DIR/'difference_images'
for p in [REGRESSION_DIR,REGRESSION_TABLES,REGRESSION_DIFFS]:p.mkdir(parents=True,exist_ok=True)

REGRESSION_PREDICTORS=['w','h','area','aspect_ratio','fill_ratio','touches_page_border','context_pixels','context_edge_density','context_dark_density','context_mean_intensity','context_intensity_std','maximum_mask_depth','horizontal_fraction','vertical_fraction','compact_fraction','preserve_fraction','remove_fraction','ambiguous_fraction','reconstruction_dependency_fraction']

def regression_latest(root,required):
 hits=[]
 if root.exists():
  for p in root.rglob('*'):
   if p.is_dir() and all((p/x).exists() for x in required):hits.append(p)
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def regression_imap(folder):
 return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}

def regression_loadz(path):
 with np.load(path) as z:return {k:z[k] for k in z.files}

def regression_digest(a):
 return hashlib.sha256(np.ascontiguousarray(a).tobytes()+str(a.shape).encode()+str(a.dtype).encode()).hexdigest()

V75_REGRESSION=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else regression_latest(V75_ROOT,['tables/cache_status.csv','tables/surrounding_area_evidence_by_component.csv'])
V77_REGRESSION=Path(V77_DEVELOPMENT_RUN_OVERRIDE) if V77_DEVELOPMENT_RUN_OVERRIDE else regression_latest(V77_ROOT,['tables/asymmetric_component_dataset.csv','v7_7_manifest.json'])
if V75_REGRESSION is None or V77_REGRESSION is None:raise RuntimeError('Complete corrected V7.5 and V7.7 development runs are required for the mandatory regression gate.')

reg_clean_map=regression_imap(ROOT/'source'/'clean')
reg_observed_map=regression_imap(ROOT/'aligned')
reg_status=pd.read_csv(V75_REGRESSION/'tables'/'cache_status.csv')
reg_cache_lookup={str(r.page):Path(r.cache) for _,r in reg_status.iterrows()}
reg_evidence=pd.read_csv(V75_REGRESSION/'tables'/'surrounding_area_evidence_by_component.csv')
reg_v77=pd.read_csv(V77_REGRESSION/'tables'/'asymmetric_component_dataset.csv')
method_col=next((x for x in ['method','candidate','geometry_candidate'] if x in reg_evidence.columns),None)
if method_col is not None:reg_evidence=reg_evidence[reg_evidence[method_col].astype(str).str.lower().str.contains('t80')].copy()
reg_pages=sorted(set(reg_clean_map)&set(reg_observed_map)&set(reg_cache_lookup))
if REGRESSION_PAGE_ALLOWLIST is not None:reg_pages=[p for p in reg_pages if p in set(REGRESSION_PAGE_ALLOWLIST)]
if REGRESSION_MAX_PAGES is not None:reg_pages=reg_pages[:REGRESSION_MAX_PAGES]
if not reg_pages:raise RuntimeError('Mandatory development regression roster is empty.')
if REGRESSION_PAGE_ALLOWLIST is not None or REGRESSION_MAX_PAGES is not None:raise RuntimeError('Mandatory pilot gate requires the complete development roster: leave REGRESSION_PAGE_ALLOWLIST and REGRESSION_MAX_PAGES as None.')
print('Mandatory development regression pages:',len(reg_pages))
print('V7.5 reference:',V75_REGRESSION)
print('V7.7 reference:',V77_REGRESSION)


In [ ]:
reg_stage_a=[];reg_missing=[];reg_geometry=[];reg_routes=[];reg_components=[];reg_predictor_rows=[];reg_labels=[]
for page in reg_pages:
 z=regression_loadz(reg_cache_lookup[page]);clean=cv2.imread(str(reg_clean_map[page]),0);obs=cv2.imread(str(reg_observed_map[page]),0)
 absent=[x for x in ['mask','telea','geometry_t80'] if x not in z]
 if absent:reg_missing.append({'page':page,'missing':'|'.join(absent)});continue
 mask=z['mask'];telea=z['telea']
 reg_stage_a.append({'page':page,'observed_shape_ok':obs is not None and clean is not None and obs.shape==clean.shape,'mask_shape_ok':mask.shape==obs.shape,'telea_shape_ok':telea.shape==obs.shape,'telea_outside_mask_immutable':bool(np.array_equal(telea[mask==0],obs[mask==0])),'mask_pixels':int((mask>0).sum())})
 built,br=geometry_t80(obs,mask,telea);ref=z['geometry_t80']
 reg_geometry.append({'page':page,'exact':bool(np.array_equal(ref,built)),'different_pixels':int(np.sum(ref!=built)),'max_abs_difference':int(np.max(np.abs(ref.astype(np.int16)-built.astype(np.int16)))),'reference_hash':regression_digest(ref),'rebuilt_hash':regression_digest(built),'reference_outside_immutable':bool(np.array_equal(ref[mask==0],obs[mask==0])),'rebuilt_outside_immutable':bool(np.array_equal(built[mask==0],obs[mask==0]))})
 diff=np.abs(ref.astype(np.int16)-built.astype(np.int16)).astype(np.uint8)
 if diff.any():cv2.imwrite(str(REGRESSION_DIFFS/f'{page}_geometry_absdiff.png'),diff)
 for key in ['preserve','remove','ambiguous','horizontal','vertical','compact']:
  rk=f'g80_{key}'
  if rk not in z:reg_routes.append({'page':page,'route':key,'tested':False,'exact':False,'xor_pixels':np.nan,'iou':np.nan})
  else:
   a=z[rk]>0;b=br[key];reg_routes.append({'page':page,'route':key,'tested':True,'exact':bool(np.array_equal(a,b)),'xor_pixels':int(np.logical_xor(a,b).sum()),'iou':float((a&b).sum()/max((a|b).sum(),1))})
 n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);ev=reg_evidence[reg_evidence.page.astype(str)==page];vref=reg_v77[reg_v77.page.astype(str)==page]
 for cid in range(1,n):
  if stats[cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
  comp=lab==cid;fv=surrounding_features(obs,comp,br);refrow=ev[ev.component.astype(int)==cid]
  reg_components.append({'page':page,'component':cid,'reference_row_count':len(refrow),'x':int(stats[cid,0]),'y':int(stats[cid,1]),'w_cc':int(stats[cid,2]),'h_cc':int(stats[cid,3]),'area_cc':int(stats[cid,4])})
  for field in REGRESSION_PREDICTORS:
   rv=refrow.iloc[0][field] if len(refrow)==1 and field in refrow.columns else np.nan;bv=fv[field]
   if isinstance(bv,(bool,np.bool_)):match=pd.notna(rv) and bool(rv)==bool(bv);delta=0 if match else 1
   else:match=pd.notna(rv) and bool(np.isclose(float(rv),float(bv),atol=REGRESSION_ATOL,rtol=REGRESSION_RTOL,equal_nan=True));delta=abs(float(rv)-float(bv)) if pd.notna(rv) else np.nan
   reg_predictor_rows.append({'page':page,'component':cid,'field':field,'reference_value':rv,'pilot_value':bv,'match':match,'absolute_difference':delta})
  lr=vref[vref.component.astype(int)==cid];reg_labels.append({'page':page,'component':cid,'v7_7_rows':len(lr)})

reg_a=pd.DataFrame(reg_stage_a);reg_missing_df=pd.DataFrame(reg_missing);reg_geometry_df=pd.DataFrame(reg_geometry);reg_route_df=pd.DataFrame(reg_routes);reg_component_df=pd.DataFrame(reg_components);reg_pred_df=pd.DataFrame(reg_predictor_rows);reg_label_df=pd.DataFrame(reg_labels)
reg_a.to_csv(REGRESSION_TABLES/'stage_a_cache_integrity.csv',index=False);reg_missing_df.to_csv(REGRESSION_TABLES/'missing_arrays.csv',index=False);reg_geometry_df.to_csv(REGRESSION_TABLES/'stage_b_geometry.csv',index=False);reg_route_df.to_csv(REGRESSION_TABLES/'stage_b_routing.csv',index=False);reg_component_df.to_csv(REGRESSION_TABLES/'stage_c_components.csv',index=False);reg_pred_df.to_csv(REGRESSION_TABLES/'stage_c_predictors.csv',index=False);reg_label_df.to_csv(REGRESSION_TABLES/'stage_d_v7_7_alignment.csv',index=False)
reg_predictor_summary=reg_pred_df.groupby('field',as_index=False).agg(comparisons=('match','size'),mismatches=('match',lambda x:int((~x).sum())),max_absolute_difference=('absolute_difference','max')).sort_values('mismatches',ascending=False)
reg_predictor_summary.to_csv(REGRESSION_TABLES/'predictor_mismatch_summary.csv',index=False)


In [ ]:
all_route_tested=bool(len(reg_route_df) and reg_route_df.tested.all())
reg_checks={
 'authoritative_arrays_complete':len(reg_missing)==0,
 'stage_a_shapes_and_telea':bool(len(reg_a) and reg_a[['observed_shape_ok','mask_shape_ok','telea_shape_ok','telea_outside_mask_immutable']].all().all()),
 'stage_b_geometry_exact':bool(len(reg_geometry_df) and reg_geometry_df.exact.all()),
 'stage_b_routing_arrays_available':all_route_tested,
 'stage_b_routing_exact':bool(all_route_tested and reg_route_df.exact.all()),
 'stage_c_component_reference_unique':bool(len(reg_component_df) and (reg_component_df.reference_row_count==1).all()),
 'stage_c_predictors_exact':bool(len(reg_pred_df) and reg_pred_df.match.all()),
 'stage_d_v7_7_component_alignment':bool(len(reg_label_df) and (reg_label_df.v7_7_rows==1).all()),
}
reg_order=list(reg_checks);reg_first_failure=next((x for x in reg_order if not reg_checks[x]),None);REGRESSION_VERDICT='PASS' if reg_first_failure is None else 'FAIL'
reg_result={'verdict':REGRESSION_VERDICT,'first_failed_stage':reg_first_failure,**reg_checks,'pages':len(reg_pages),'components':len(reg_component_df),'geometry_different_pixels':int(reg_geometry_df.different_pixels.sum()) if len(reg_geometry_df) else None,'routing_xor_pixels':int(reg_route_df.xor_pixels.fillna(0).sum()) if len(reg_route_df) else None,'predictor_mismatches':int((~reg_pred_df.match).sum()) if len(reg_pred_df) else None,'v7_5_reference':str(V75_REGRESSION),'v7_7_reference':str(V77_REGRESSION),'canonical_structural_block_sha256':CANONICAL_STRUCTURAL_BLOCK_SHA256}
pd.DataFrame([reg_result]).to_csv(REGRESSION_TABLES/'audit_verdict.csv',index=False)
(REGRESSION_DIR/'regression_manifest.json').write_text(json.dumps(reg_result,indent=2))
print(json.dumps(reg_result,indent=2))
if REGRESSION_VERDICT!='PASS':raise RuntimeError(f'PILOT REVIEWER BLOCKED: mandatory development regression failed at {reg_first_failure}. See {REGRESSION_DIR}')
print('MANDATORY DEVELOPMENT REGRESSION PASSED. Pilot input processing may proceed.')


## Build proposals and component evidence


In [ ]:

def md5_file(path):
 h=hashlib.md5()
 with open(path,'rb') as f:
  for block in iter(lambda:f.read(1048576),b''):h.update(block)
 return h.hexdigest()

def read_audit_records(folder,label):
 records=[];parse_failures=[]
 for path in sorted(Path(folder).glob('*.json')):
  try:
   value=json.loads(path.read_text())
   if not isinstance(value,dict):raise ValueError('audit root must be an object')
   value['_audit_path']=str(path);records.append(value)
  except Exception as error:parse_failures.append({'audit_type':label,'audit_path':str(path),'error':str(error)})
 return records,parse_failures

REVIEW_AUDITS,review_parse_failures=read_audit_records(REVIEW_AUDIT_DIR,'review')
DRAFT_AUDITS,draft_parse_failures=read_audit_records(DRAFT_AUDIT_DIR,'draft')
AUDIT_PARSE_FAILURES=review_parse_failures+draft_parse_failures
if AUDIT_PARSE_FAILURES:
 pd.DataFrame(AUDIT_PARSE_FAILURES).to_csv(TABLES/'hitl_audit_parse_failures.csv',index=False)
 raise RuntimeError(f'HITL audit lookup blocked: {len(AUDIT_PARSE_FAILURES)} unreadable audit record(s). See hitl_audit_parse_failures.csv')

def matches_by_hash(records,source_md5):return [r for r in records if str(r.get('source_md5','')).lower()==source_md5.lower()]
def required_path(record,key,audit_type):
 raw=record.get(key)
 if not raw:raise RuntimeError(f'{audit_type} audit {record.get("_audit_path")} is stale: missing {key}.')
 p=Path(raw)
 if not p.exists():raise RuntimeError(f'{audit_type} audit {record.get("_audit_path")} is stale: referenced {key} does not exist: {p}')
 return p

def resolve_hitl(source):
 source_md5=md5_file(source);review_matches=matches_by_hash(REVIEW_AUDITS,source_md5);draft_matches=matches_by_hash(DRAFT_AUDITS,source_md5)
 if len(review_matches)>1:raise RuntimeError(f'{source.name}: duplicate review audits for source_md5 {source_md5}: {[x["_audit_path"] for x in review_matches]}')
 if len(draft_matches)>1:raise RuntimeError(f'{source.name}: duplicate draft audits for source_md5 {source_md5}: {[x["_audit_path"] for x in draft_matches]}')
 review=review_matches[0] if review_matches else None;draft=draft_matches[0] if draft_matches else None
 if review is None and draft is None:raise RuntimeError(f'{source.name}: no HITL review or draft audit matches source_md5 {source_md5}. Run Notebook 06 first.')
 confidence_path=None;draft_mask_path=None;draft_audit_path=None
 if draft is not None:
  draft_audit_path=draft['_audit_path'];draft_mask_path=required_path(draft,'draft_mask','draft');confidence_path=required_path(draft,'confidence_map','draft')
  if draft.get('source_file') and Path(draft['source_file']).exists() and md5_file(Path(draft['source_file']))!=source_md5:raise RuntimeError(f'{source.name}: draft audit source_file hash is stale.')
 observed_path=source;mask_path=draft_mask_path;mask_status='unreviewed_draft';review_audit_path=None
 if review is not None:
  review_audit_path=review['_audit_path'];observed_path=required_path(review,'aligned_output','review');mask_path=required_path(review,'corrected_mask_output','review');mask_status='reviewed_corrected'
  if review.get('source') and Path(review['source']).exists() and md5_file(Path(review['source']))!=source_md5:raise RuntimeError(f'{source.name}: review audit source hash is stale.')
 observed=cv2.imread(str(observed_path),0);mask=cv2.imread(str(mask_path),0)
 if observed is None or mask is None:raise RuntimeError(f'{source.name}: resolved observed page or mask is unreadable.')
 if observed.shape!=mask.shape:raise RuntimeError(f'{source.name}: resolved observed/mask shape mismatch: {observed.shape} vs {mask.shape}.')
 confidence=None
 if confidence_path is not None:
  confidence=np.load(confidence_path).astype(np.float32)
  if confidence.shape!=observed.shape:raise RuntimeError(f'{source.name}: confidence map shape {confidence.shape} does not match resolved observed page {observed.shape}.')
 else:confidence=(mask>0).astype(np.float32)*0.20
 return {'source_original':source,'source_md5':source_md5,'observed_path':observed_path,'mask_path':mask_path,'confidence_path':confidence_path,'review_audit_path':review_audit_path,'draft_audit_path':draft_audit_path,'mask_status':mask_status,'observed':observed,'seed':mask,'confidence':confidence}

def build_page(source):
 resolved=resolve_hitl(source);observed=resolved['observed'];seed=resolved['seed'];confidence=resolved['confidence']
 mask=prepare_mask(seed,2);telea=telea_reconstruct(observed,mask);proposal,routes=geometry_t80(observed,mask,telea)
 n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);rows=[]
 for cid in range(1,n):
  if stats[cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
  comp=lab==cid;row={'page':source.stem,'component':cid,'x':int(stats[cid,0]),'y':int(stats[cid,1]),'bbox_w':int(stats[cid,2]),'bbox_h':int(stats[cid,3]),'annotation_confidence_mean':float(confidence[comp].mean()) if comp.any() else 0.0,'annotation_confidence_max':float(confidence[comp].max()) if comp.any() else 0.0};row.update(surrounding_features(observed,comp,routes));rows.append(row)
 frame=pd.DataFrame(rows)
 for old,new in LEGACY_ALIASES.items():
  if old in predictors and old not in frame:frame[old]=frame[new]
 missing=[x for x in predictors if x not in frame]
 if missing:raise RuntimeError(f'Missing frozen predictors: {missing}')
 X=frame[predictors].apply(pd.to_numeric,errors='coerce');raw=model.predict_proba(X)[:,1];prob=calibrator.predict(raw);frame['raw_structural_safe_score']=raw;frame['prob_structural_safe']=prob;frame['passes_frozen_structural_threshold']=prob>=threshold
 red=(frame['ambiguous_fraction']>0.50)|((frame['preserve_fraction']>0.20)&~frame['passes_frozen_structural_threshold']);frame['pilot_category']=np.where(frame['passes_frozen_structural_threshold'],'green',np.where(red,'red','amber'))
 cv2.imwrite(str(PROPOSALS/f'{source.stem}.png'),proposal);cv2.imwrite(str(MASK_OUT/f'{source.stem}_authoritative.png'),mask);np.savez_compressed(ROUTING/f'{source.stem}.npz',mask=mask,telea=telea,geometry_t80=proposal,**{f'g80_{k}':v.astype(np.uint8) for k,v in routes.items() if k in ['preserve','remove','ambiguous','horizontal','vertical','compact']})
 resolution={k:(str(v) if isinstance(v,Path) else v) for k,v in resolved.items() if k not in ['observed','seed','confidence','source_original']};resolution['source_original']=str(source);resolution['observed_sha256']=sha256_file(resolved['observed_path']);resolution['authoritative_mask_sha256']=sha256_file(resolved['mask_path']);resolution['confidence_sha256']=sha256_file(resolved['confidence_path']) if resolved['confidence_path'] else None
 (AUDIT/f'{source.stem}_input_resolution.json').write_text(json.dumps(resolution,indent=2))
 return {'source':source,'source_md5':resolved['source_md5'],'observed':observed,'mask':mask,'proposal':proposal,'routes':routes,'labels':lab,'components':frame,'mask_source':str(resolved['mask_path']),'confidence_source':str(resolved['confidence_path']) if resolved['confidence_path'] else None,'observed_source':str(resolved['observed_path']),'mask_status':resolved['mask_status'],'review_audit':resolved['review_audit_path'],'draft_audit':resolved['draft_audit_path']}

sources=sorted(p for p in INCOMING.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED)
if not sources:raise RuntimeError(f'Put pilot pages in {INCOMING}')
PAGES={p.name:build_page(p) for p in sources};all_components=pd.concat([x['components'] for x in PAGES.values()],ignore_index=True);all_components.to_csv(TABLES/'component_evidence.csv',index=False)
print(all_components.groupby(['page','pilot_category']).size())


## Reviewer V3: compact diagnostic, queue navigation and conservative decision preservation

The structural and regression-tested prefix above is unchanged. This reviewer adds seed-versus-dilation diagnostics, first/next-unreviewed navigation, status-labelled component choices, an original-image mask editor, and exact preservation of decisions for components unaffected by an edit.


In [ ]:
if globals().get('REGRESSION_VERDICT')!='PASS':raise RuntimeError('PILOT REVIEWER BLOCKED: mandatory development regression must PASS.')
PROTECTED_PREFIX_SHA256='6cf870d5ef07964951decdab89df119d43c781f0b595ac993c561eec6d8f34e2';DECISIONS=RUN/'decisions';EDITS=RUN/'edited_masks'
for p in [DECISIONS,EDITS]:p.mkdir(parents=True,exist_ok=True)
STATE={}
def uid(p,r,c):return f'{Path(p).stem}:r{r}:c{int(c)}'
def fresh_decisions(frame,p,r):return {uid(p,r,x.component):{'status':'unreviewed','notes':'','history':[]} for _,x in frame.iterrows()}
for p,z in PAGES.items():STATE[p]={'revision':0,'observed':z['observed'].copy(),'seed':cv2.imread(z['mask_source'],0),'mask':z['mask'].copy(),'proposal':z['proposal'].copy(),'routes':z['routes'],'labels':z['labels'].copy(),'components':z['components'].copy(),'decisions':fresh_decisions(z['components'],p,0),'superseded':[],'context':{k:z.get(k) for k in ['source_md5','mask_source','confidence_source','observed_source','mask_status','review_audit','draft_audit']}}
def persist(p,event):
 s=STATE[p];(DECISIONS/f'{Path(p).stem}_review_state.json').write_text(json.dumps({'page':p,'revision':s['revision'],'policy_id':policy['policy_id'],'threshold':threshold,'protected_prefix_sha256':PROTECTED_PREFIX_SHA256,'decisions':s['decisions'],'superseded':s['superseded'],'context':s['context'],'event':event,'updated_utc':datetime.now(timezone.utc).isoformat()},indent=2,default=str))
def array_sig(a):return hashlib.sha256(np.ascontiguousarray(a).tobytes()+str(a.shape).encode()+str(a.dtype).encode()).hexdigest()
def component_signature(labels,cid,proposal,routes):
 m=labels==int(cid);ys,xs=np.where(m)
 if not len(xs):return None
 x0,x1=xs.min(),xs.max()+1;y0,y1=ys.min(),ys.max()+1
 parts=[m[y0:y1,x0:x1].astype(np.uint8),proposal[y0:y1,x0:x1]]+[routes[k][y0:y1,x0:x1].astype(np.uint8) for k in ['preserve','remove','ambiguous']]
 return (int(x0),int(y0),int(x1),int(y1),*(array_sig(a) for a in parts))
def snapshot_components(s,p):
 out={}
 for _,r in s['components'].iterrows():
  cid=int(r.component);out[component_signature(s['labels'],cid,s['proposal'],s['routes'])]=(cid,s['decisions'][uid(p,s['revision'],cid)].copy())
 return out
def rescore(p,seed,reason):
 s=STATE[p];old=snapshot_components(s,p);obs=s['observed'];seed=np.where(seed>0,255,0).astype(np.uint8);mask=prepare_mask(seed,2);telea=telea_reconstruct(obs,mask);proposal,routes=geometry_t80(obs,mask,telea);n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);cp=s['context'].get('confidence_source');conf=np.load(cp).astype(np.float32) if cp else (seed>0).astype(np.float32)*.20;rows=[]
 for cid in range(1,n):
  if stats[cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
  comp=lab==cid;row={'page':Path(p).stem,'component':cid,'x':int(stats[cid,0]),'y':int(stats[cid,1]),'bbox_w':int(stats[cid,2]),'bbox_h':int(stats[cid,3]),'annotation_confidence_mean':float(conf[comp].mean()),'annotation_confidence_max':float(conf[comp].max())};row.update(surrounding_features(obs,comp,routes));rows.append(row)
 frame=pd.DataFrame(rows)
 for a,b in LEGACY_ALIASES.items():
  if a in predictors and a not in frame:frame[a]=frame[b]
 missing=[x for x in predictors if x not in frame]
 if missing:raise RuntimeError(f'Missing frozen predictors after edit: {missing}')
 X=frame[predictors].apply(pd.to_numeric,errors='coerce');raw=model.predict_proba(X)[:,1];prob=calibrator.predict(raw);frame['raw_structural_safe_score']=raw;frame['prob_structural_safe']=prob;frame['passes_frozen_structural_threshold']=prob>=threshold;red=(frame.ambiguous_fraction>.50)|((frame.preserve_fraction>.20)&~frame.passes_frozen_structural_threshold);frame['pilot_category']=np.where(frame.passes_frozen_structural_threshold,'green',np.where(red,'red','amber'))
 nr=s['revision']+1;new=fresh_decisions(frame,p,nr);preserved=0;matched_old=set()
 for _,r in frame.iterrows():
  cid=int(r.component);sig=component_signature(lab,cid,proposal,routes)
  if sig in old:
   oldcid,d=old[sig];new[uid(p,nr,cid)]={**d,'history':list(d.get('history',[]))+[{'event':'preserved_after_mask_edit','from_revision':s['revision'],'from_component':oldcid,'utc':datetime.now(timezone.utc).isoformat()}]};preserved+=1;matched_old.add(sig)
 for sig,(oldcid,d) in old.items():
  if sig not in matched_old:s['superseded'].append({'uid':uid(p,s['revision'],oldcid),**d,'status':'superseded','superseded_utc':datetime.now(timezone.utc).isoformat()})
 s.update({'revision':nr,'seed':seed,'mask':mask,'proposal':proposal,'routes':routes,'labels':lab,'components':frame,'decisions':new});cv2.imwrite(str(EDITS/f'{Path(p).stem}_r{nr}.png'),seed);persist(p,{'event':'edit_rescore','reason':reason,'preserved_decisions':preserved,'new_components':len(frame)});return frame,preserved
def counts(p):
 v=[x['status'] for x in STATE[p]['decisions'].values()];return {k:v.count(k) for k in ['unreviewed','accepted','left_unchanged','deferred']}
def current(p):
 s=STATE[p];o=s['observed'].copy()
 for _,r in s['components'].iterrows():
  if s['decisions'][uid(p,s['revision'],r.component)]['status']=='accepted':m=s['labels']==int(r.component);o[m]=s['proposal'][m]
 assert np.array_equal(o[s['mask']==0],s['observed'][s['mask']==0]);return o
def tint(base,layers):
 rgb=cv2.cvtColor(base,cv2.COLOR_GRAY2RGB) if base.ndim==2 else base.copy();lay=rgb.copy();a=np.zeros(rgb.shape[:2],np.float32)
 for m,col,op in layers:lay[m]=col;a[m]=np.maximum(a[m],op)
 return (rgb*(1-a[...,None])+lay*a[...,None]).astype(np.uint8)
def outline(p,base,cat='all',status='all'):
 s=STATE[p];rgb=cv2.cvtColor(base,cv2.COLOR_GRAY2RGB);cols={'green':(40,180,65),'amber':(245,165,0),'red':(225,40,40)}
 for _,r in s['components'].iterrows():
  st=s['decisions'][uid(p,s['revision'],r.component)]['status']
  if cat!='all' and r.pilot_category!=cat or status!='all' and st!=status:continue
  col=cols[r.pilot_category];cv2.rectangle(rgb,(int(r.x),int(r.y)),(int(r.x+r.bbox_w),int(r.y+r.bbox_h)),col,2);cv2.putText(rgb,f'C{int(r.component)}',(int(r.x),max(13,int(r.y)-4)),cv2.FONT_HERSHEY_SIMPLEX,.42,col,1,cv2.LINE_AA)
 return rgb
def overview(p,cat='all',status='all',view='current'):
 s=STATE[p];return outline(p,{'current':current(p),'proposal':s['proposal'],'original':s['observed']}[view],cat,status)
def component_diagnostic(p,c,show_seed=False,show_dilation=False,show_changed=False,show_routing=False,margin=90):
 s=STATE[p]
 if c is None:return overview(p),'Select a component.'
 q=s['components'][s['components'].component.astype(int)==int(c)]
 if q.empty:return overview(p),'Stale component after edit.'
 r=q.iloc[0];comp=s['labels']==int(c);seed=s['seed']>0;dilation=(s['mask']>0)&~seed;layers=[]
 if show_seed:layers.append((comp&seed,(230,45,45),.58))
 if show_dilation:layers.append((comp&dilation,(255,220,20),.62))
 if show_changed:layers.append((comp&(s['proposal']!=s['observed']),(220,30,220),.72))
 if show_routing:layers += [(comp&s['routes']['preserve'],(40,120,255),.65),(comp&s['routes']['remove'],(35,190,90),.65),(comp&s['routes']['ambiguous'],(255,145,20),.65)]
 proposed=tint(s['proposal'],layers) if layers else cv2.cvtColor(s['proposal'],cv2.COLOR_GRAY2RGB);H,W=s['observed'].shape;x0=max(0,int(r.x)-margin);y0=max(0,int(r.y)-margin);x1=min(W,int(r.x+r.bbox_w)+margin);y1=min(H,int(r.y+r.bbox_h)+margin);orig=cv2.cvtColor(s['observed'][y0:y1,x0:x1],cv2.COLOR_GRAY2RGB);prop=proposed[y0:y1,x0:x1];div=np.full((orig.shape[0],7,3),235,np.uint8);pair=np.hstack([orig,div,prop]);off=orig.shape[1]+7;col={'green':(40,180,65),'amber':(245,165,0),'red':(225,40,40)}[r.pilot_category]
 for xoff in [0,off]:cv2.rectangle(pair,(xoff+int(r.x)-x0,int(r.y)-y0),(xoff+int(r.x+r.bbox_w)-x0,int(r.y+r.bbox_h)-y0),col,2)
 cv2.putText(pair,'ORIGINAL',(6,17),cv2.FONT_HERSHEY_SIMPLEX,.46,(30,30,30),1,cv2.LINE_AA);cv2.putText(pair,'PROPOSED',(off+6,17),cv2.FONT_HERSHEY_SIMPLEX,.46,(30,30,30),1,cv2.LINE_AA)
 st=s['decisions'][uid(p,s['revision'],c)]['status'];vals={'seed':int(np.sum(comp&seed)),'dilation_only':int(np.sum(comp&dilation)),'removal_mask':int(comp.sum()),'preserve':int(np.sum(comp&s['routes']['preserve'])),'remove':int(np.sum(comp&s['routes']['remove'])),'ambiguous':int(np.sum(comp&s['routes']['ambiguous'])),'changed':int(np.sum(comp&(s['proposal']!=s['observed'])))}
 info=f"C{c} | {r.pilot_category.upper()} | {st} | seed {vals['seed']} | dilation-only {vals['dilation_only']} | removal {vals['removal_mask']} | preserve {vals['preserve']} | remove {vals['remove']} | ambiguous {vals['ambiguous']} | changed {vals['changed']} | p_safe {r.prob_structural_safe:.4f}"
 return pair,info
def queue_choices(p):
 s=STATE[p];result=[]
 for _,r in s['components'].iterrows():
  cid=int(r.component);st=s['decisions'][uid(p,s['revision'],cid)]['status'];result.append((f'C{cid} [{st}]',cid))
 return result
def first_unreviewed(p):
 s=STATE[p]
 for _,r in s['components'].iterrows():
  cid=int(r.component)
  if s['decisions'][uid(p,s['revision'],cid)]['status']=='unreviewed':return cid
 return None
def table(p):
 s=STATE[p];return pd.DataFrame([{'component':int(r.component),'category':r.pilot_category,'status':s['decisions'][uid(p,s['revision'],r.component)]['status'],'area_pixels':int(r.area),'p_structural_safe':round(float(r.prob_structural_safe),4),'ambiguous_fraction':round(float(r.ambiguous_fraction),4)} for _,r in s['components'].iterrows()])
def decide(p,c,status,notes):
 if c is None:raise gr.Error('Select a component.')
 s=STATE[p];k=uid(p,s['revision'],c);s['decisions'][k].update({'status':status,'notes':notes,'utc':datetime.now(timezone.utc).isoformat()});s['decisions'][k]['history'].append({'status':status,'notes':notes});persist(p,{'event':'decision','component':int(c),'status':status});return f'C{c}: {status}'
def editor_value(p):
 s=STATE[p];rgba=np.zeros((*s['seed'].shape,4),np.uint8);rgba[s['seed']>0]=(235,45,45,150);return {'background':cv2.cvtColor(s['observed'],cv2.COLOR_GRAY2RGB),'layers':[rgba],'composite':None}
def decode_editor(v,shape):
 if not isinstance(v,dict):raise gr.Error('Mask editor did not return layer data.')
 layers=v.get('layers') or []
 if not layers:return np.zeros(shape,np.uint8)
 out=np.zeros(shape,bool)
 for layer in layers:
  a=np.asarray(layer)
  if a.shape[:2]!=shape:a=cv2.resize(a,(shape[1],shape[0]),interpolation=cv2.INTER_NEAREST)
  if a.ndim==3 and a.shape[2]>=4:out|=a[:,:,3]>20
  elif a.ndim==3:out|=np.max(a[:,:,:3],axis=2)>30
  else:out|=a>30
 return out.astype(np.uint8)*255
def edit_mask(p,v,notes):
 s=STATE[p];seed=decode_editor(v,s['observed'].shape)
 if np.array_equal(seed,s['seed']):raise gr.Error('Edited mask is unchanged.')
 old=s['revision'];f,preserved=rescore(p,seed,notes);return f'Revision {old} to {s["revision"]}; {len(f)} components rescored; {preserved} unchanged decision(s) preserved.'
def approve(p,notes):
 c=counts(p)
 if c['unreviewed'] or c['deferred']:raise gr.Error(f'Page not ready: {c["unreviewed"]} unreviewed and {c["deferred"]} deferred.')
 s=STATE[p];f=current(p);assert np.array_equal(f[s['mask']==0],s['observed'][s['mask']==0]);path=FINAL/f'{Path(p).stem}.png';cv2.imwrite(str(path),f);persist(p,{'event':'final_approval','notes':notes,'path':str(path)});return f,f'Approved final page: {path}'
def refresh(p,cat,st,view,preferred=None):
 s=STATE[p];choices=queue_choices(p);valid=[v for _,v in choices];chosen=preferred if preferred in valid else first_unreviewed(p);c=counts(p);summary=f'Revision {s["revision"]} | total {len(valid)} | unreviewed {c["unreviewed"]} | deferred {c["deferred"]} | accepted {c["accepted"]} | unchanged {c["left_unchanged"]}'
 return overview(p,cat,st,view),table(p),gr.update(choices=choices,value=chosen),editor_value(p),current(p),summary
CSS=""".gradio-container{max-width:100vw!important;padding:6px!important}.compact img{max-height:58vh!important;object-fit:contain!important}.compact-table{max-height:28vh!important;overflow:auto!important}.form{gap:5px!important}footer{display:none!important}"""
pages=list(STATE);first=pages[0]
with gr.Blocks(title='Pilot 09 Compact Reviewer',css=CSS) as app:
 with gr.Row(equal_height=True):page=gr.Dropdown(pages,value=first,label='Page',scale=2);category=gr.Radio(['all','green','amber','red'],value='all',label='Category',scale=3);status_filter=gr.Radio(['all','unreviewed','deferred','accepted','left_unchanged'],value='all',label='State',scale=4);view=gr.Radio(['current','proposal','original'],value='current',label='Image',scale=3)
 summary=gr.Textbox(label='Status',interactive=False)
 with gr.Tabs():
  with gr.Tab('Overview'):ov=gr.Image(height=480,label='Current/proposed correction with component outlines',elem_classes=['compact']);qt=gr.Dataframe(interactive=False,label='Queue',elem_classes=['compact-table'])
  with gr.Tab('Component review'):
   with gr.Row():component=gr.Dropdown(label='Component [status]',scale=3);show_seed=gr.Checkbox(False,label='Seed mask (red)');show_dilation=gr.Checkbox(False,label='Dilation-only (yellow)');show_changed=gr.Checkbox(False,label='Changed (magenta)');show_routing=gr.Checkbox(False,label='Routing')
   ci=gr.Image(height=430,label='Original | proposed',elem_classes=['compact']);info=gr.Textbox(interactive=False,label='Pixel diagnostic');notes=gr.Textbox(label='Notes',lines=1)
   with gr.Row():accept=gr.Button('Accept',variant='primary');leave=gr.Button('Leave unchanged');defer=gr.Button('Defer');allgreen=gr.Button('Confirm all green')
  with gr.Tab('Edit mask and rescore'):
   gr.Markdown('**Brush key:** red = annotation included in seed mask; eraser = remove annotation from seed mask. Background = original observed page. Diagnostic colours after rescore: yellow dilation-only, blue preserve, green remove, orange ambiguous, magenta actual change.')
   editor=gr.ImageEditor(type='numpy',label='Original page with editable red seed-mask overlay',height=520,brush=gr.Brush(colors=['#eb2d2d'],color_mode='fixed'),eraser=gr.Eraser());edit_notes=gr.Textbox(label='Edit reason',lines=1);edit=gr.Button('Apply edit and rescore',variant='primary');edit_status=gr.Textbox(interactive=False)
  with gr.Tab('Final approval'):cur=gr.Image(height=480,label='Current reviewed result',elem_classes=['compact']);approval_notes=gr.Textbox(label='Approval notes',lines=1);approve_btn=gr.Button('Approve final page',variant='primary');final=gr.Image(height=420,label='Approved final page');approval_status=gr.Textbox(interactive=False)
 outs=[ov,qt,component,editor,cur,summary]
 def refresh_default(p,cat,st,v):return refresh(p,cat,st,v)
 for ctl in [page,category,status_filter,view]:ctl.change(refresh_default,[page,category,status_filter,view],outs)
 app.load(refresh_default,[page,category,status_filter,view],outs)
 comp_inputs=[page,component,show_seed,show_dilation,show_changed,show_routing]
 component.change(component_diagnostic,comp_inputs,[ci,info])
 for toggle in [show_seed,show_dilation,show_changed,show_routing]:toggle.change(component_diagnostic,comp_inputs,[ci,info])
 def decision_refresh(p,c,n,status,cat,st,v,sm,sd,sc,sr):
  msg=decide(p,c,status,n);vals=refresh(p,cat,st,v);nextc=vals[2]['value'] if isinstance(vals[2],dict) else first_unreviewed(p);pair,detail=component_diagnostic(p,nextc,sm,sd,sc,sr);return msg,*vals,pair,detail
 decision_inputs=[page,component,notes,category,status_filter,view,show_seed,show_dilation,show_changed,show_routing]
 decision_outputs=[info,*outs,ci,info]
 accept.click(lambda p,c,n,cat,st,v,sm,sd,sc,sr:decision_refresh(p,c,n,'accepted',cat,st,v,sm,sd,sc,sr),decision_inputs,decision_outputs)
 leave.click(lambda p,c,n,cat,st,v,sm,sd,sc,sr:decision_refresh(p,c,n,'left_unchanged',cat,st,v,sm,sd,sc,sr),decision_inputs,decision_outputs)
 defer.click(lambda p,c,n,cat,st,v,sm,sd,sc,sr:decision_refresh(p,c,n,'deferred',cat,st,v,sm,sd,sc,sr),decision_inputs,decision_outputs)
 def greens(p,n,cat,st,v):
  k=0
  for c in STATE[p]['components'].loc[STATE[p]['components'].pilot_category=='green','component']:
   if STATE[p]['decisions'][uid(p,STATE[p]['revision'],c)]['status'] in ['unreviewed','deferred']:decide(p,int(c),'accepted',n);k+=1
  return f'Confirmed {k} green component(s).',*refresh(p,cat,st,v)
 allgreen.click(greens,[page,notes,category,status_filter,view],[info,*outs])
 def edit_refresh(p,m,n,cat,st,v):return edit_mask(p,m,n),*refresh(p,cat,st,v)
 edit.click(edit_refresh,[page,editor,edit_notes,category,status_filter,view],[edit_status,*outs]);approve_btn.click(approve,[page,approval_notes],[final,approval_status])
app.launch(share=PUBLIC_LINK,debug=False,show_error=True)


## Decision-preservation contract

A decision is preserved across a mask edit only when the old component and a new component have an exact unique signature match: identical bounding box, effective component mask, proposed corrected pixels, and preserve/remove/ambiguous routing pixels. Components that split, merge, move or change reconstruction are returned to `unreviewed`; unmatched old decisions remain in superseded history.


In [ ]:
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'Pilot09-6-compact-diagnostic-state-preserving','supervised_only':True,'auto_accept_enabled':False,'protected_prefix_sha256':PROTECTED_PREFIX_SHA256,'policy_id':policy['policy_id'],'policy_threshold':threshold,'development_regression':reg_result,'automatic_next_unreviewed':True,'status_labelled_component_dropdown':True,'seed_vs_dilation_diagnostic':True,'original_image_mask_editor':True,'decision_preservation_rule':'unique exact component signature: bbox, effective mask, proposal, preserve/remove/ambiguous routes','approval_requires_no_unreviewed_or_deferred':True};(RUN/'pilot_manifest.json').write_text(json.dumps(manifest,indent=2,default=str));print('Pilot manifest:',RUN/'pilot_manifest.json')
